# Project III

In [130]:
%load_ext autoreload
%autoreload 2
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2
from scipy.stats import norm


import w8_estimation as est 
import w8_LinearModel as lm
import w8_probit as probit
import w8_logit as logit

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [131]:
dat = pd.read_csv('ppcs_cc.csv')

In [132]:
dat.sample(5)

,sblack,shisp,swhite,sother,smale,sage,sempl,sincome,spop,daytime,inctype_lin,omajblack,omajhisp,omajwhite,omajother,osplit,sbehavior,year,anyuseofforce_coded
586,0,0,1,0,1,51,1,3,1,0,2,0,0,1,0,0,0,2011,0
1633,0,0,1,0,0,45,1,1,1,1,2,0,0,1,0,0,0,2011,0
3414,0,0,1,0,0,63,1,1,1,1,2,0,0,1,0,0,0,2011,0
1577,0,0,1,0,0,39,1,3,1,1,2,0,0,1,0,0,0,2011,0
2852,0,0,1,0,0,49,1,2,1,1,2,0,0,1,0,0,0,2011,0


In [133]:
dat.describe()

,sblack,shisp,swhite,sother,smale,sage,sempl,sincome,spop,daytime,inctype_lin,omajblack,omajhisp,omajwhite,omajother,osplit,sbehavior,year,anyuseofforce_coded
count,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.000000,3799.0,3799.000000,3799.0,3799.000000
mean,0.110555,0.101606,0.739142,0.048697,0.529613,41.010003,0.695446,2.164780,1.362727,0.666491,1.958410,0.060805,0.023954,0.903659,0.011582,0.0,0.065017,2011.0,0.005001
std,0.313622,0.302169,0.439160,0.215262,0.499188,16.146916,0.460279,0.848262,0.765598,0.471529,0.199676,0.239005,0.152925,0.295097,0.107009,0.0,0.246589,0.0,0.070552
min,0.000000,0.000000,0.000000,0.000000,0.000000,16.000000,0.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,2011.0,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,27.000000,0.000000,1.000000,1.000000,0.000000,2.000000,0.000000,0.000000,1.000000,0.000000,0.0,0.000000,2011.0,0.000000
50%,0.000000,0.000000,1.000000,0.000000,1.000000,40.000000,1.000000,2.000000,1.000000,1.000000,2.000000,0.000000,0.000000,1.000000,0.000000,0.0,0.000000,2011.0,0.000000
75%,0.000000,0.000000,1.000000,0.000000,1.000000,52.000000,1.000000,3.000000,1.000000,1.000000,2.000000,0.000000,0.000000,1.000000,0.000000,0.0,0.000000,2011.0,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,90.000000,1.000000,3.000000,4.000000,1.000000,2.000000,1.000000,1.000000,1.000000,1.000000,0.0,1.000000,2011.0,1.000000


dat[dat['anyuseofforce_coded'] == 1].describe()

In [134]:
df=dat.copy()

df.drop(columns=['osplit', 'year'], inplace=True) #no variance in these variables, so we drop them


df = pd.get_dummies(df, columns=['sincome', 'spop', 'inctype_lin'], dtype=int) #make dummy variables for categorical variables

#add age squared variable
#df['sage_sq'] = df['sage']**2
df['const'] = 1

#choose reference categories for categorical variables
df = df.drop(columns=['swhite', 'omajwhite', 'sincome_2', 'spop_2', 'inctype_lin_2', 'sage']) #drop reference categories



df.insert(0, 'const', df.pop('const'))

df.sample(5)

,const,sblack,shisp,sother,smale,sage,sempl,daytime,omajblack,omajhisp,omajother,sbehavior,anyuseofforce_coded,sincome_1,sincome_3,spop_1,spop_3,spop_4,inctype_lin_1,sage_sq
2245,1,0,0,0,0,49,1,0,0,0,0,0,0,1,0,1,0,0,0,2401
1036,1,0,0,0,1,22,1,1,0,0,0,0,0,0,0,0,0,0,0,484
1901,1,0,0,0,1,32,1,0,1,0,0,0,0,1,0,0,1,0,1,1024
2208,1,0,0,0,0,36,1,1,0,0,0,0,0,0,0,1,0,0,0,1296
3688,1,0,1,0,0,34,1,1,0,0,0,0,0,0,1,1,0,0,0,1156


In [135]:
labels_x = list(df.columns)
labels_x.remove('anyuseofforce_coded') #remove dependent variable from list of labels
labels_y = ['anyuseofforce_coded'] #dependent variable

#
y = df.anyuseofforce_coded.values
#x = df[['const', 'sblack', 'shisp', 'sother', 'smale']].values
x = df[labels_x].values

K = x.shape[1] #number of independent variables
N = x.shape[0] #number of observations

print(x)

[[   1    1    0 ...    0    0  324]
 [   1    1    0 ...    1    0  400]
 [   1    1    0 ...    0    0  484]
 ...
 [   1    0    0 ...    0    0 5776]
 [   1    0    0 ...    1    0 6241]
 [   1    0    0 ...    0    0 5625]]


# LPM

In [136]:
ols_results = lm.estimate(y=y, x=x, robust_se=True)
ols_tab = lm.print_table((labels_y, labels_x), ols_results, title='LPM')
ols_tab

LPM
Dependent variable: ['anyuseofforce_coded']

R2 = 0.034
sigma2 = nan


,b_hat,se,t
const,0.0119,0.0097,1.2363
sblack,0.0018,0.0045,0.3892
shisp,0.0087,0.0059,1.4803
sother,-0.0006,0.0059,-0.1029
smale,0.0043,0.0023,1.8735
sage,0.0001,0.0003,0.3120
sempl,-0.0061,0.0035,-1.7327
daytime,-0.0018,0.0028,-0.6272
omajblack,-0.0055,0.0017,-3.1888
omajhisp,-0.0013,0.0096,-0.1344


# PROBIT

In [137]:
theta0 = probit.starting_values(y=y, x=x)
print(theta0)

[ 2.98304867e-02  4.39080561e-03  2.17894822e-02 -1.50653554e-03
  1.06662699e-02  2.67012777e-04 -1.53206083e-02 -4.40326101e-03
 -1.36558596e-02 -3.22514721e-03 -1.91342929e-02  8.81066512e-02
 -3.71296528e-03  1.07614614e-03 -2.02077247e-02 -8.37718712e-03
  8.16536559e-03  6.98073193e-02 -7.40149819e-06]


In [138]:
probit_results = est.estimate(probit.q, theta0, y=y, x=x, method='BFGS')

Optimization terminated successfully.
         Current function value: 0.092128
         Iterations: 1
         Function evaluations: 40
         Gradient evaluations: 2


LinAlgError: Singular matrix

In [ ]:
probit_tab = est.print_table((labels_x), probit_results, title='PROBIT')
probit_tab